# 01-项目介绍：沉默模式解码器

本 notebook 介绍 Silence Pattern Decoder 项目的基本概念和使用方法。

## 什么是沉默模式解码器？

沉默模式解码器是一个社会科学研究工具，使用计算方法分析投票行为，特别关注**弃权模式**（沉默/弃权）。通过分析这些"沉默"数据，我们可以检测：

- **隐式公识**（Implicit Consensus）：群体中未明确表达但普遍认同的观点
- **少数派影响**（Minority Influence）：少数群体对多数群体决策的潜在影响
- **群体压力**（Group Pressure）：群体动态如何抑制个人表达

## 安装和设置

首先导入必要的库并设置路径：

In [ ]:
import sys
import os

# 如果需要，添加项目路径
project_path = os.path.abspath('..')
if project_path not in sys.path:
    sys.path.insert(0, project_path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams

# 设置中文显示
rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
rcParams['axes.unicode_minus'] = False

# 设置图表风格
plt.style.use('seaborn-v0_8-whitegrid')
rcParams['figure.figsize'] = (10, 6)

print("库导入成功！")

## 核心概念

### 投票行为模型

我们的模型基于以下几个关键概念：

1. **信念维度**（Belief Dimensions）：每个代理具有多维信念向量
2. **影响容忍度**（Influence Tolerance）：代理接受他人影响的倾向
3. **意见强度**（Opinion Strength）：代理坚持自己观点的程度
4. **投票历史**（Voting History）：代理过往的投票记录

### 弃权模式

代理可能因为以下原因选择弃权：
- 对所有候选人都不满意
- 感受到群体压力
- 信息不足
- 意见强度低

### 社会影响网络

代理之间存在影响关系，形成一个有向图：
- 边的权重表示影响强度
- 影响是逐轮传播的
- 影响可以改变代理的信念

In [ ]:
# 导入项目模块
from silence_decoder.src.agent import Agent
from silence_decoder.src.influence import InfluenceGraph
from silence_decoder.src.voting import VotingSystem
from silence_decoder.src.simulation import SimulationEngine
from silence_decoder.src.pattern_detector import PatternDetector
from silence_decoder.src.analysis import compute_abstention_rate

print("所有模块导入成功！")

## 快速开始：运行一个简单模拟

下面是一个完整的模拟示例：

In [ ]:
# 创建模拟引擎
engine = SimulationEngine(
    num_agents=50,          # 50个代理
    num_candidates=3,       # 3个候选人
    num_belief_dimensions=2, # 2维信念空间
    seed=42                 # 固定随机种子
)

# 生成代理
agents = engine.generate_random_agents()
print(f"生成了 {len(agents)} 个代理")

# 生成影响图
influence_graph = engine.generate_random_influence_graph(density=0.2)
print(f"影响图包含 {len(influence_graph.edges)} 条边")

In [ ]:
# 运行模拟
result = engine.run_simulation(
    num_rounds=20,
    agents=agents,
    influence_graph=influence_graph,
    voting_rule='approval',
    influence_strength=0.3,
    belief_threshold=0.5,
    verbose=False
)

# 查看结果摘要
print(f"模拟完成：{result.num_rounds} 轮，{result.num_agents} 个代理")
print(f"最终弃权率: {result.final_abstention_rate:.2%}")
print(f"平均弃权率: {result.avg_abstention_rate:.2%}")
print(f"公识分数: {result.consensus_score:.3f}")

## 可视化结果

让我们绘制弃权率随时间的变化：

In [ ]:
from silence_decoder.src.visualizer import plot_abstention_timeline

# 获取弃权率趋势
abstention_rates = engine.get_abstention_trend(result)

# 绘制时间线
fig = plot_abstention_timeline(abstention_rates, show=True)
plt.title('弃权率随时间变化')
plt.show()

## 检测弃权模式

使用 PatternDetector 检测潜在的群体动力学模式：

In [ ]:
# 创建模式检测器
detector = PatternDetector()

# 收集投票数据
voting_data = {
    'abstention_rates': [r.abstention_rate for r in result.rounds],
    'vote_distributions': [r.vote_distribution for r in result.rounds],
    'belief_means': [r.belief_mean for r in result.rounds]
}

# 检测模式
consensus_result = detector.detect_consensus(voting_data)
minority_result = detector.detect_minority_pressure(voting_data)
oppression_result = detector.detect_oppression(voting_data)

print("模式检测结果：")
print(f"  公识检测: 分数={consensus_result[0]:.3f}, 置信度={consensus_result[1]:.3f}")
print(f"  少数派影响: 分数={minority_result[0]:.3f}, 置信度={minority_result[1]:.3f}")
print(f"  压迫检测: 分数={oppression_result[0]:.3f}, 置信度={oppression_result[1]:.3f}")

## 下一步

现在你已经了解了基本概念，可以：

1. 运行更复杂的模拟（更多代理、更多轮次）
2. 修改参数观察不同行为
3. 查看 `02-consensus-detection.ipynb` 学习公识检测
4. 查看 `03-influence-analysis.ipynb` 学习影响分析

## API 参考

### SimulationEngine 主要方法
- `generate_random_agents(num_agents, num_belief_dimensions)` - 生成随机代理
- `generate_random_influence_graph(agents, density)` - 生成影响图
- `run_simulation(num_rounds, ...)` - 运行完整模拟
- `run_batch_simulation(num_simulations, sim_params)` - 批量运行模拟

### PatternDetector 主要方法
- `detect_consensus(voting_data)` - 检测公识
- `detect_minority_pressure(voting_data)` - 检测少数派影响
- `detect_oppression(voting_data)` - 检测压迫